# exercises Chapter 2: Array-Based Lists

## Exercise 2.1. add_all()

The List method add all(i, c) inserts all elements of the Collection c 
into the list at position i. (The add(i, x) method is a special case
where c = {x}.) 

Explain why, for the data structures in this chapter, it is not 
efficient to implement add all(i, c) by repeated calls to add(i, x). 
Design and implement a more efficient implementation.

In [1476]:
# By repeating calls to add(i, x), we would also need to do
# repeated calls to resize() or balance() (depending in DS)
# To avoid this this, it would be better to merge current
# current arrays(s) with a call to resize() once.


# Example for ArrayStack:

from typing import Any
import copy


class ArrayStack:
    def __init__(self):
        self.a = []
        self.n = 0       # number of items (len)

    def get(self, i:int):
        return self.a[i]

    def set(self, i:int, x:Any) -> Any:
        y = copy.copy(self.a[i])
        self.a[i] = x
        return y

    def add(self, i:int, x:Any):
        if self.n == len(self.a):
            self.resize()

        # shift right: go backwards
        for j in range(self.n, i, -1):
            self.a[j] = self.a[j-1]

        self.a[i] = x
        self.n += 1

    def remove(self, i):
        x = self.a[i]

        # shift left
        for j in range(i, self.n - 1):
            self.a[j] = self.a[j + 1]

        # leave a None at the end
        self.a[self.n - 1] = None

        self.n -= 1

        if len(self.a) >= 3 * self.n:
            self.resize()

        return x

    def resize(self):
        # ensure at least size 1
        new_capacity = max(1, 2*self.n)
        b = [None] * new_capacity
        for i in range(self.n):
            b[i] = self.a[i]

        self.a = b

    def add_all(self, i:int, c:list):
        m = len(c)
        if m == 0:
            return

        # ensure capacity
        if self.n + m > len(self.a):
            new_capacity = max(1, 2*(self.n + m))
            b = [None] * new_capacity

            # copy left part
            for j in range(i):
                b[j] = self.a[j]

            # copy c
            for j in range(m):
                b[i + j] = c[j]

            # copy right part
            for j in range(i, self.n):
                b[j + m] = self.a[j]

            self.a = b
            self.n += m
            return

        # enough space: shift right
        # backwards to not overwrite data
        for j in range(self.n - 1, i - 1, -1):
            self.a[j + m] = self.a[j]

        # insert c
        for j in range(m):
            self.a[i + j] = c[j]

        self.n += m



In [1477]:
my_as = ArrayStack()
my_as.add(0, "a")
my_as.add(1, "b")
my_as.add(2, "c")
my_as.add(3, "d")
my_as.add(4, "e")

print(my_as.a)

['a', 'b', 'c', 'd', 'e', None, None, None]


In [1478]:
my_as.add_all(i=2, c=["1", "2", "3", "4"])
print(my_as.a)
print(my_as.n)

['a', 'b', '1', '2', '3', '4', 'c', 'd', 'e', None, None, None, None, None, None, None, None, None]
9


## Exercise 2.2. RandomQueue

Design and implement a RandomQueue. This is an implementation of the Queue interface in which the remove() operation removes an element that is chosen uniformly at random among all the elements currently in the queue. (Think of a RandomQueue as a bag in which we can add § elements or reach in and blindly remove some random element.) 

The add(x) and remove() operations in a RandomQueue should
run in constant time per operation.

In [1479]:
import random
from math import ceil

class RandomQeue:
    def __init__(self):
        self.a = []
        self.n = 0

    def add(self, x):
        if self.n == len(self.a):
            self.resize()
        self.a[self.n] = x
        self.n += 1

    def remove(self):
        r = random.randrange(self.n)
        x = self.a[r]
        self.a[r] = self.a[self.n - 1]
        self.a[self.n - 1] = None
        self.n -= 1

        if self.n < ceil(len(self.a) / 3):
            self.resize()

        return x

    def resize(self):
        new = [None] * max(1, 2*self.n)
        for i in range(self.n):
            new[i] = self.a[i]
        self.a = new

In [1480]:
my_rq = RandomQeue()
my_rq.add("a")
my_rq.add("b")
my_rq.add("c")
my_rq.add("d")

print(my_rq.remove())
print(my_rq.a)

b
['a', 'd', 'c', None]


In [1481]:
my_rq.add("f")
my_rq.add("g")
my_rq.add("h")
my_rq.add("i")
my_rq.add("j")

print(my_rq.a)

['a', 'd', 'c', 'f', 'g', 'h', 'i', 'j']


In [1482]:
print(my_rq.remove())
print(my_rq.remove())
print(my_rq.remove())
print(my_rq.remove())
print(my_rq.remove())
print(my_rq.remove())
print(my_rq.a)

j
d
h
c
f
a
['g', 'i', None, None]


## Exercise 2.3. Trequeue 

Design and implement a Treque (triple-ended queue). This is a List implementation in which get(i) and set(i, x) run in constant time and add(i, x) and remove(i) run in time O(1 + min{i, n − i, |n/2 − i|}) .

In other words, modifications are fast if they are near either end or near
the middle of the list.

In [ ]:
class Trequeue:
    def __init__(self):
        self.left  = []
        self.mid   = []
        self.right = []

    def set(self, i:int, x:Any) -> None:
        self.validate_i(i)
        if i <= len(self.left) - 1:
            self.left[i] = x

        elif i <= len(self.mid) - 1:
            self.mid[i - len(self.left)] = x

        else:
            self.right[i - (len(self.left) + len(self.mid))] = x

    def get(self, i:int) -> Any:
        self.validate_i(i)
        if i <= len(self.left) - 1:
            return self.left[i]

        elif i <= len(self.mid) - 1:
            return self.mid[i - len(self.left)]

        else:
            return self.right[i - (len(self.left) + len(self.mid))]

    def add(self, i:int, x:Any) -> None:
        pass

    def remove(self, i:int) -> Any:
        pass

    def validate_i(self, i):
        i_end = sum([len(self.left), len(self.mid), len(self.right)]) - 1
        if i_end < 0:
            raise ValueError("Container has no values!")
        if not 0 <= i < i_end:
            raise IndexError(f"Index must be in range: 0 - {i_end}, got {i}")


    def __str__(self):
        return f"{my_trq.left + my_trq.mid + my_trq.right}"


In [1484]:
my_trq = Trequeue()
my_trq.left = ["a"]
my_trq.mid = ["b"]
my_trq.right = ["c"]

In [1485]:
my_trq.set(1, "Q")

print(my_trq)

['a', 'Q', 'c']
